# Notebook 04 - Feature Engineering & Preprocessing

## Objective

The objective of this notebook is to transform the raw dataset into a machine-learning-ready format.

This notebook covers:

- Feature selection
- Missing value treatment
- Encoding categorical variables
- Scaling numerical variables
- Building a reusable preprocessing pipeline

The final output will be a preprocessing pipeline that can be reused during both model training and deployment.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
import sys

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

import joblib

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [4]:
from src.data.loader import load_train_data
from src.features.preprocessor import DataPreprocessor

In [5]:
train = load_train_data()

df = train.copy()

In [6]:
train = load_train_data()

df = train.copy()

In [7]:
X = df.drop(columns="health_condition")

y = df["health_condition"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [9]:
X_train = X_train.drop(columns=["id"])

X_test = X_test.drop(columns=["id"])

In [10]:
preprocessor = DataPreprocessor()

pipeline = preprocessor.build_pipeline(X_train)

In [11]:
num_cols = X_train.select_dtypes(

    include=np.number

).columns.tolist()

cat_cols = X_train.select_dtypes(

    exclude=np.number

).columns.tolist()

In [12]:
X_train_processed = pipeline.fit_transform(X_train)

X_test_processed = pipeline.transform(X_test)

In [13]:
print(X_train_processed.shape)

print(X_test_processed.shape)

(552070, 25)
(138018, 25)


In [14]:
encoded_cols = (
    pipeline.named_transformers_["categorical"]
            .named_steps["encoder"]
            .get_feature_names_out(cat_cols)
)

all_features = num_cols + list(encoded_cols)

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=all_features,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=all_features,
    index=X_test.index
)

In [15]:
joblib.dump(
    pipeline,
    "../artifacts/preprocessor.pkl"
)

['../artifacts/preprocessor.pkl']

# Summary

The preprocessing pipeline was successfully created.

The following preprocessing steps were applied:

- Identifier column removed
- Missing values imputed
- Numerical variables standardised
- Categorical variables one-hot encoded

The fitted preprocessing pipeline has been saved and will be reused during model training and deployment.

This ensures that identical preprocessing is applied to both training data and unseen data.

In [16]:
print("Missing Values in Training Set :", X_train_processed.isnull().sum().sum())
print("Missing Values in Test Set     :", X_test_processed.isnull().sum().sum())

Missing Values in Training Set : 0
Missing Values in Test Set     : 0


In [17]:
display(X_train_processed.head())

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type_balanced,diet_type_non-veg,diet_type_veg,stress_level_high,stress_level_low,stress_level_medium,sleep_quality_average,sleep_quality_good,sleep_quality_poor,physical_activity_level_active,physical_activity_level_moderate,physical_activity_level_sedentary,smoking_alcohol_no,smoking_alcohol_occasional,smoking_alcohol_yes,gender_female,gender_male,gender_other
313415,0.15,1.99,1.58,1.22,-1.86,0.70,-0.31,0.00,1.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00
3515,-0.00,-0.00,0.51,0.46,1.25,0.82,-0.71,1.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00
501194,1.45,0.90,-0.64,0.26,0.22,-1.05,1.80,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00
303602,-0.00,-0.07,-0.16,-0.19,-0.40,-1.17,-0.53,0.00,0.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00
117943,1.60,-0.85,-0.40,-0.36,1.26,0.96,0.32,0.00,1.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00


In [18]:
print(f"Original Features : {X_train.shape[1]}")
print(f"Processed Features: {X_train_processed.shape[1]}")

Original Features : 13
Processed Features: 25


In [19]:
# Save processed training data
X_train_processed.to_csv(
    "../artifacts/X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "../artifacts/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../artifacts/y_train.csv",
    index=False
)

y_test.to_csv(
    "../artifacts/y_test.csv",
    index=False
)

In [20]:
from src.data.dataset_manager import DatasetManager

dataset_manager = DatasetManager()

dataset_manager.save_processed_data(
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

✅ Processed datasets saved successfully.
